In [1]:
# ===============================
# INSTALL (if needed)
# ===============================
# !pip install xgboost openpyxl

# ===============================
# IMPORTS
# ===============================
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, f1_score

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# ===============================
# LOAD DATA
# ===============================
df = pd.read_excel("../data/final_cleaned_dataset.xlsx")

# ===============================
# CREATE 2-CLASS TARGET
# ===============================
def simplify_target_2(label):
    if label == "Probably not":
        return "No Risk"
    else:
        return "Risk"

df["target_2class"] = df["target"].apply(simplify_target_2)

print("Target distribution:")
print(df["target_2class"].value_counts())

# ===============================
# PREPARE X AND y
# ===============================
X = df.drop(columns=["target", "target_2class"], errors="ignore").astype(int)
y = df["target_2class"]

# ===============================
# ENCODE TARGET
# ===============================
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print("Classes:", list(le.classes_))

# ===============================
# FIRST SPLIT
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# ===============================
# FEATURE SELECTION (TOP 30)
# ===============================
temp_model = XGBClassifier(
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss"
)

temp_model.fit(X_train, y_train)

importance = pd.Series(temp_model.feature_importances_, index=X_train.columns)
top_features = importance.sort_values(ascending=False).head(30).index.tolist()

print("Top features selected:", len(top_features))

X_top = X[top_features]

# ===============================
# NEW SPLIT WITH TOP FEATURES
# ===============================
X_train, X_test, y_train, y_test = train_test_split(
    X_top,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

# ===============================
# MODELS
# ===============================
models_2class = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=2000, class_weight="balanced"))
    ]),
    
    "SVM": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC(class_weight="balanced"))
    ]),
    
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ),
    
    "XGBoost": XGBClassifier(
        objective="binary:logistic",
        n_estimators=200,
        learning_rate=0.1,
        max_depth=5,
        random_state=42,
        eval_metric="logloss"
    )
}

# ===============================
# TRAIN + COMPARE
# ===============================
results = []

for name, model in models_2class.items():
    print(f"\n===== {name} =====")
    
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average="macro")
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "F1_macro": f1
    })
    
    print("Accuracy:", acc)
    print("F1 Macro:", f1)
    print(classification_report(y_test, y_pred, target_names=le.classes_))

# ===============================
# RESULTS TABLE
# ===============================
results_df_2class = pd.DataFrame(results).sort_values(by="F1_macro", ascending=False)

print("\n=== FINAL RESULTS ===")
print(results_df_2class)

# ===============================
# SAVE BEST MODEL
# ===============================
best_model_name = results_df_2class.iloc[0]["Model"]
best_model = models_2class[best_model_name]

best_model.fit(X_train, y_train)

joblib.dump(best_model, "best_model_2class.pkl")
joblib.dump(le, "label_encoder_2class.pkl")

print(f"\nBest model ({best_model_name}) saved ✅")

# ===============================
# SAVE RESULTS
# ===============================
results_df_2class.to_excel("model_comparison_2class.xlsx", index=False)
print("Results saved ✅")

Target distribution:
target_2class
Risk       430
No Risk    120
Name: count, dtype: int64
Classes: ['No Risk', 'Risk']
Top features selected: 30

===== Logistic Regression =====
Accuracy: 0.6363636363636364
F1 Macro: 0.5592948717948718
              precision    recall  f1-score   support

     No Risk       0.30      0.50      0.38        24
        Risk       0.83      0.67      0.74        86

    accuracy                           0.64       110
   macro avg       0.56      0.59      0.56       110
weighted avg       0.71      0.64      0.66       110


===== SVM =====
Accuracy: 0.6818181818181818
F1 Macro: 0.5946941783345616
              precision    recall  f1-score   support

     No Risk       0.34      0.50      0.41        24
        Risk       0.84      0.73      0.78        86

    accuracy                           0.68       110
   macro avg       0.59      0.62      0.59       110
weighted avg       0.73      0.68      0.70       110


===== Random Forest =====
Accurac

In [ ]:
selected_features = [
    "Days_FeltFat_13-15 days",
    "Days_WeightAffectedSelfJudgment_13-15 days",
    "Days_ShapeAffectedSelfJudgment_13-15 days",
    "Days_DissatisfiedWithWeight_13-15 days",
    "Days_TriedLimitFoodControlShapeOrWeight_13-15 days",
    "Days_FearLosingControlOverEating_13-15 days"
]

In [1]:
importance.sort_values(ascending=False).head(6)

NameError: name 'importance' is not defined